[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/ml/04-machine-learning/ml-ethics.ipynb)

# Ethics in Machine Learning

*AIBits Academy · Machine Learning End To End · Responsible ML · New*

The previous chapter asked "is this model fair across groups?" This one asks two different questions: "is this model's reported performance honest?" and "does the training data actually license the conclusions being drawn from it?"

**How to use this notebook:** run the cells top to bottom (Runtime → Run all). Each code cell is the same code you saw on the course page, so you can compare your output with the lesson. The graded exercises are at the end; try them before opening the solutions.

*Interactive animations and quiz cards stay on the course page.*

*Small numeric differences from the lesson page are normal: library versions, random seeds and dataset copies change the last digits. The conclusions should agree.*

> **Scope — How This Differs From the Fairness Chapter**
>
> ML Fairness & Bias Auditing covered group-level fairness metrics, proxy variables, and mitigation techniques. This chapter covers two separate ethical failure modes that can occur even in a model that passes every fairness audit: **misleading performance claims** (a benchmarking-integrity problem) and **training data that doesn't support the conclusions drawn from it** (a data-provenance problem).

## Misleading Performance Claims

A model's reported accuracy can be technically true and still misleading — not through fabrication, but through **selective reporting**. The most common version: trying many things (models, feature sets, random seeds, train/test splits) and reporting only the best result, without disclosing how many attempts were made. This is the same "garden of forking paths" problem that affects p-hacking in statistics, applied to model evaluation.

The following demonstration uses **the same model and the same data** for all 50 trials — only the random train/test split changes each time:

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

# A genuinely weak signal — features are barely informative (class_sep=0.5, 35% label noise)
X, y = make_classification(n_samples=400, n_features=20, n_informative=3,
                            class_sep=0.5, flip_y=0.35, random_state=0)

accs = []
for trial in range(50):
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=trial)
    model = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
    accs.append(model.score(X_te, y_te))

accs = np.array(accs)
print(f"Mean test accuracy across 50 splits : {accs.mean():.4f}")
print(f"Best single split observed          : {accs.max():.4f}")
print(f"Inflation from reporting only the best: {accs.max()-accs.mean():.4f}")

The model and the data never changed — only which 30% of rows happened to land in the test set. Reporting "58% accuracy" from the lucky split, without mentioning 49 other splits averaged closer to 48%, is not a lie in any single sentence, yet it systematically misrepresents what the model will actually do on new data. The honest fix is simple and already covered in this course: report the **cross-validated mean and standard deviation**, not a single best-looking number — and disclose how many configurations were tried before landing on the reported one.

| Malpractice pattern | Why it misleads | Honest alternative |
|---|---|---|
| Reporting best-of-N splits/seeds | Inflates the number by however much random variation the metric has | Report mean ± std across k-fold CV, disclose N attempts |
| Comparing against a weak baseline | Makes ordinary performance look like a breakthrough | Compare against the strongest reasonable baseline, not a strawman |
| Reporting accuracy on imbalanced data | A 99% "accurate" model can be catching almost no positives (see Model Evaluation page) | Report precision/recall/F1 or PR-AUC alongside accuracy |
| Tuning hyperparameters using the test set | The "test" set is no longer unseen — reported score is really a training score | Reserve a validation set (or nested CV) for all tuning; touch the test set exactly once |

## Limitations of Training Data

A model's predictions are only as trustworthy as the licence its training data provides for making them. Three distinct ways that licence can be weaker than it looks:

| Limitation | What it means | Example |
|---|---|---|
| Non-representativeness | The training sample doesn't match the population the model will actually see in production | A model trained only on metro-city Flipkart orders may not generalise to Tier-2/3 city buying patterns once deployed nationally |
| Labels as an imperfect proxy | The target column measures something correlated with, but not identical to, what the model is actually meant to predict | "Loan defaulted" measures defaults *among approved applicants only* — it says nothing about applicants who were rejected and never got the chance to repay or default (a form of selection bias baked permanently into the label) |
| Feedback loops | The model's own past predictions influenced what data got collected next, compounding any initial bias | A fraud model that never flags a certain transaction pattern means that pattern is never investigated, so the training label for it stays "not fraud" indefinitely — regardless of the truth |

> **⚠ "The Data Doesn't Speak for Itself"**
>
> A dataset cannot tell you whether it's representative of the deployment population, whether its labels were collected fairly, or whether a feedback loop has been quietly distorting it for years — these are questions that require knowing *how* the data was collected, not just what's in it. Treating a dataset as ground truth without that provenance context is itself the ethical lapse, independent of any bias found (or missed) by a fairness audit.

## A Responsible ML Checklist

Pulling together tools already covered across this course into a single pre-deployment checklist:

- **Performance:** reported via cross-validated mean ± std, not a cherry-picked best split (this chapter); appropriate metric for the class balance (Handling Imbalanced Data, Model Evaluation)
- **Explainability:** can a rejected/flagged individual be given a real reason? (Model Interpretability & Explainability)
- **Fairness:** audited across relevant groups using an explicitly chosen metric (ML Fairness & Bias Auditing)
- **Calibration:** if probability values feed a downstream cost calculation, they've been checked, not just the ranking (Model Evaluation)
- **Data provenance:** known limitations of the training sample and labels are documented, not assumed away (this chapter)
- **Monitoring:** a plan exists to detect data/concept drift after deployment, not just at launch (Model Evaluation)

### ❓ Conceptual Q&A

---
## Graded exercises

Each exercise has a **starter cell** you complete and a **check cell** that prints ✅ or ❌. The solution is folded away underneath — try first.

In [ ]:
# --- self-check helper (used by the exercises) ---------------------------------------------
def check(name, ok):
    print(("\u2705 " if ok else "\u274c ") + name)


### Exercise 1 · Easy · Beat the baseline, not just chance

With 70% negatives, a model that always says "negative" is 70% accurate. Compute `baseline` (majority-class accuracy of `y_te`) and `model_acc` (a `LogisticRegression` on the training split), and `adds_value` = `model_acc > baseline + 0.02`.

In [ ]:
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
X, y = make_classification(n_samples=3000, n_features=6, n_informative=2, weights=[0.7, 0.3], class_sep=0.5, flip_y=0.3, random_state=0)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, random_state=0)
baseline = model_acc = adds_value = None   # TODO


In [ ]:
try:
    check("baseline near 0.7", 0.6 < baseline < 0.8)
    check("flag is a bool", adds_value in (True, False))
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
X, y = make_classification(n_samples=3000, n_features=6, n_informative=2, weights=[0.7, 0.3], class_sep=0.5, flip_y=0.3, random_state=0)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, random_state=0)
baseline = max(y_te.mean(), 1 - y_te.mean())
model_acc = LogisticRegression().fit(X_tr, y_tr).score(X_te, y_te)
adds_value = bool(model_acc > baseline + 0.02)

```

</details>

### Exercise 2 · Medium · An honest error bar on accuracy

A model got 84 of 100 test predictions right. Compute the 95% **Wilson** interval for the true accuracy and store `(lo, hi)` in `ci` (`statsmodels.stats.proportion.proportion_confint(..., method="wilson")`).

In [ ]:
from statsmodels.stats.proportion import proportion_confint
ci = None   # TODO


In [ ]:
try:
    check("interval brackets 0.84", ci[0] < 0.84 < ci[1])
    check("about +-7 points wide", 0.11 < ci[1] - ci[0] < 0.17)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
from statsmodels.stats.proportion import proportion_confint
ci = proportion_confint(84, 100, alpha=0.05, method="wilson")

```

</details>

### Exercise 3 · Stretch · Small samples, wide intervals

Compute the width of the same 84% accuracy interval for test-set sizes `n` = 50, 500, 5000 (successes = 0.84 * n). Store the widths in a dict `widths` keyed by `n`.

In [ ]:
widths = {}   # TODO


In [ ]:
try:
    check("three sizes", set(widths) == {50, 500, 5000})
    check("more data narrows the interval", widths[50] > widths[500] > widths[5000])
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
widths = {}
for n in (50, 500, 5000):
    lo, hi = proportion_confint(round(0.84 * n), n, method="wilson")
    widths[n] = hi - lo

```

Before promising a business 84% accuracy, report the uncertainty: with 50 test cases the honest claim is 'somewhere between the low 70s and low 90s'.

</details>

---
*Back to the course: **Machine Learning End To End → Ethics in Machine Learning**.*